In [ ]:
import numpy as np
import matplotlib.pyplot as plt   
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.pardir)))
from main import load_cifar_batch  

Data Preparation

In [ ]:
np.random.seed(42)
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
x_train,y_train = load_cifar_batch('../datasets/cifar-10-batches-py/data_batch_2')

#removed 500 samples of images for validation   
x_train = x_train[500:]
y_train = y_train[500:]

# from (9500,32,32,3) to (9500,3072)
x_train = x_train.reshape(x_train.shape[0],-1) 
x_norm = x_train/255. 
print(f'x_train after norm the pixels: {x_norm[0]}')
#hot-coded lab
y_coded = np.eye(10)[y_train] 
print(f'one-hot encoded labels for our 9500 images {y_coded}')

#validation sets images -> (500,1)  labels->500
x_val = x_train[:500]
y_val = y_train[:500]
x_val = x_val.reshape(x_val.shape[0],-1) 
val_norm = x_val/255.
print(f'x_val after norm the pixels: {val_norm[0]}')

W = np.random.randn(len(class_names), x_norm.shape[1]) * 0.01 
b = np.zeros(shape=(1, 10))

In [ ]:
# --- 4. LOSS FUNCTION ---
'''
compute loss function for the softmax classifier.

X -> (N, D) where N is the number of images and D is the number of features (3072 for CIFAR-10)
W -> (C, D) where C is the number of classes (10 for CIFAR-10)
b -> (1, C) bias term for each class
y_tr -> (N,) true labels for each image, where each label is an integer in the range [0, C-1]
'''

def compute_loss(X,W,b,y_tr):
    
    N = X.shape[0]
    
    #raw scores for all the images
    scores = np.dot(X,W.T) + b
    
    #normalized scores with softmax 
    scores -= np.max(scores,axis=1,keepdims=True)
    exp = np.exp(scores) 
    prob = exp / np.sum(exp,axis=1,keepdims=True)
    
    
    #for each image we have the corresponding label trueY   
    correct_class_prob = prob[np.arange(N),y_tr]
    
    #added 1e-15 to avoid zero inside log 
    losses = -np.log(correct_class_prob + 1e-15)
    
    total_loss = np.sum(losses) / N
    
    return total_loss

In [ ]:
def compute_grad(X,W,b,reg,y_true):

    N = X.shape[0]
    
    scores = np.dot(X,W.T) + b
    
    scores -= np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    z = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    
    # Convert integer labels to one-hot
    if y_true.ndim == 1:
        y_one_hot = np.eye(10)[y_true]
    else:
        y_one_hot = y_true
    
    
    # Calculate the error (derivative of loss w.r.t scores) -> log(exp/sum(exp)) -> log(exp(z)) - log(sum(exp(z))) -> z - 1/sum(exp(z)) <- t_label
    dz = z - y_one_hot 
    
    dl_dw = np.dot(dz.T,X)
    dl_db = np.sum(dz, axis=0, keepdims=True)
    
    dl_dw/= N
    dl_db/= N
    
    dl_dw += reg * W
    return dl_dw,dl_db


In [ ]:
def gradient_descent(X: np.ndarray, W: np.ndarray, b: np.ndarray, y_true: np.ndarray,
                     epochs: int = 10,
                     learning_rate: float = 1e-3,
                     batch_size: int = 512
) -> (np.ndarray, np.ndarray, list):
    
    hist = []
    N = X.shape[0]
     
    for epoch in range(epochs):
        
        #taking random indices for making our model learn not only from one pattern        
        indices = np.random.permutation(N)
        X_shuffled = X[indices]
        y_shuffled = y_true[indices]
        
        epoch_loss = 0  # Track loss per epoch
        num_batches = 0
        
        for  i in range(0,N,int(batch_size)):
            X_batch = X_shuffled[i:i+int(batch_size)]
            y_batch = y_shuffled[i:i+int(batch_size)]
            
            dl_dw,dl_db = compute_grad(X_batch, W, b, reg=1e-4 ,y_true=y_batch) 
            loss = compute_loss(X_batch, W, b, y_batch)
            
            hist.append(loss)
            
            epoch_loss += loss 
            num_batches += 1
            
            
            W -= learning_rate * dl_dw            
            b -= learning_rate * dl_db      
            # print(f'W norm: {np.linalg.norm(W):.4f}, b norm: {np.linalg.norm(b):.4f}, Loss: {loss:.4f}')
            # print("dl_dw mean:", np.mean(np.abs(dl_dw)), "dl_db mean:", np.mean(np.abs(dl_db)))

        # Print epoch summary
        avg_loss = epoch_loss / num_batches
        print(f'Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}')    

        #print W and b as well to see how they decrease over time 
    
    return W,b,hist
        
        

final_w,final_b ,hist = gradient_descent(x_norm,W,b,y_train,epochs=10,learning_rate=1e-3,batch_size=512)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hist)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True)
plt.show()

# Check if loss is decreasing
print(f'First loss: {hist[0]:.4f}')
print(f'Last loss: {hist[-1]:.4f}')
print(f'Improved: {hist[0] - hist[-1]:.4f}')

In [ ]:
# Calculate moving average (smoothed loss)
window_size = 37  # ~1 epoch worth (9500 / 256 ≈ 37 batches)
moving_avg = np.convolve(hist, np.ones(window_size)/window_size, mode='valid')

plt.figure(figsize=(12, 5))
plt.plot(hist, alpha=0.3, label='Actual Loss (Noisy)')
plt.plot(moving_avg, linewidth=2, label='Moving Average (Smoothed)')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss: Noisy vs Smoothed')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

np.random.seed(42)
# add momentum and AdamW for better convergence and faster training
def gradient_descent_mom(X, W, b, y_true, 
                     epochs=10,
                     learning_rate=1e-3,
                     rho = 0.90,
                     reg = 1e-5,
                     batch_size=256
):
    N = X.shape[0]
    hist = []
    learning_hist = []
    dw_hist = []
    initial_lr = learning_rate
    vx = np.zeros_like(W)
    
    
    for epoch in range(epochs):
        
        #taking random indices for making our model learn not only from one pattern        
        indices = np.random.permutation(N)
        X_shuffled = X[indices]
        y_shuffled = y_true[indices]
        
        epoch_loss = 0  # Track loss per epoch
        num_batches = 0

        #update learning rate with cosine annealing
        current_learning_rate = initial_lr * 0.5 * (1 + np.cos(epoch*np.pi / epochs))
        learning_hist.append(current_learning_rate)
        
        for i in range(0,N,int(batch_size)):
            #mixed up the data for each batch to make sure we are not learning from one pattern
            
            X_batch = X_shuffled[i:i+int(batch_size)]
            y_batch = y_shuffled[i:i+int(batch_size)]
            
            
            dl_dw,dl_db = compute_grad(X_batch, W, b, reg ,y_true=y_batch) 
            loss = compute_loss(X_batch, W, b, y_batch)
            
            hist.append(loss)
            epoch_loss += loss 
            num_batches += 1
            
            #add velocity(build momentum) to our learning for trying to get global min
            vx = rho * vx + dl_dw
            W -= current_learning_rate * vx          
            b -= current_learning_rate * dl_db 

        print(current_learning_rate)    
        dw_hist.append(np.mean(np.abs(dl_dw)))  # Track the magnitude of the gradient
        # Print epoch summary
        avg_loss = epoch_loss / num_batches
        print(f"Epoch {epoch+1}/{epochs} | dl_dw mean: {np.mean(np.abs(dl_dw)):.2e} | dl_db mean: {np.mean(np.abs(dl_db)):.2e} | Avg Loss: {avg_loss:.4f}")    
        #print W and b as well to see how they decrease over time 
        
    return W,b,hist,learning_hist,dw_hist


In [ ]:
# --- Clean training and prediction example ---
# np.random.seed(42)
# Re-initialize weights and biases
W1 = np.random.randn(len(class_names), x_norm.shape[1]) * 0.01
b1 = np.zeros(shape=(1, 10))

# Train with a reasonable learning rate and regularization
final_w, final_b, hist, learning_hist, dw_hist = gradient_descent_mom(
    x_norm, W1, b1, y_train,
    epochs=50,
    learning_rate=1e-2,
    rho=0.93,
    reg=1e-5,    
    batch_size= 68
)
print(final_w)
# Plot loss curve
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.plot(hist[:1000])
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.subplot(1, 3, 2)

plt.plot(learning_hist)
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Over Time')
plt.grid(True)
plt.subplot(1, 3, 3)
plt.plot(dw_hist)
plt.xlabel('Epoch')
plt.ylabel('Gradient Magnitude')
plt.title('Gradient Magnitude Over Time')

plt.show()



In [ ]:
# Predict on a validation image
img_idx = 15
# Change this to test other images
scores = np.dot(val_norm[img_idx], final_w.T) + final_b
scores -= np.max(scores)  # For numerical stability

print(f'Scores for each class: {scores}')

pred_class = class_names[np.argmax(scores)]
# Show the image and prediction
img = (val_norm[img_idx] * 255).reshape(32, 32, 3).astype(np.uint8)
plt.imshow(img)
plt.title(f'Predicted: {pred_class}')
plt.axis('off')
plt.show()
loss = compute_loss(val_norm,final_w,final_b,y_val)
print(f'Validation loss: {loss}')
print(scores)

In [ ]:
from PIL import Image
img = Image.open('datasets/cifar-10-batches-py/dogo.jpg').convert('RGB')
print(img.size)
img_resize = img.resize((32, 32))
print(img_resize.size)

img_array = np.array(img_resize).reshape(1, -1) / 255.0
print(img_array)
img_flatten = img_array.reshape(1, -1)
print(f'Image shape after flattening: {img_flatten}')
print(f'Image shape after flattening: {img_flatten.shape}')

scores = np.dot(img_flatten, final_w.T) + final_b
print(f'Scores for the dog image: {scores}')
pred_class = class_names[np.argmax(scores)]
plt.imshow(img_resize)
plt.title(f'Predicted: {pred_class}')   

class_names

In [ ]:
x_test,y_test = load_cifar_batch('./datasets/cifar-10-batches-py/test_batch')
x_flat = x_test[450].flatten()
x_flat = x_flat / 255.

x_flat_re = x_flat.reshape(-1,32,32,3)
score = np.dot(x_flat,final_w.T) + final_b 
index = np.argmax(score)
print(class_names[index])

plt.imshow(x_flat_re[0])
plt.show()

In [ ]:
re = img_array.reshape(-1,32,32,3)
flat_im = re.flatten()
score1 = np.dot(flat_im,final_w.T) + final_b
index1 = class_names[np.argmax(score1)]
print(index1)
f = flat_im.reshape(-1,32,32,3)
print(f[0].shape)
# score1
plt.imshow(f[0])

So why did the Linear Classifier fail on CIFAR-10? 

In [ ]:
# Visualize the learned weights for each class 
def visualize_weights(W):
    
    classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
    plt.figure(figsize=(15, 5))
    
    for i in range(10):
        # 1. Get the weights for class i
        w_i = W[i, :] 
        
        # 2. Rescale to 0-255 for visualization
        w_min, w_max = np.min(w_i), np.max(w_i)
        w_img = 255.0 * (w_i - w_min) / (w_max - w_min)
        
        # 3. Reshape back to image dimensions (32, 32, 3)
        # Note: Depending on your data loading, it might be (3, 32, 32)
        w_img = w_img.reshape(32, 32, 3).astype(np.uint8)
        
        plt.subplot(2, 5, i + 1)
        plt.imshow(w_img)
        plt.title(classes[i])
        plt.axis('off')
    
    plt.show()

# Run this after your training is done
visualize_weights(final_w)